# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
# Print dataset metadata summary
print(f"{getattr(metadata, 'name', 'Unknown Name')}: {getattr(metadata, 'description', 'No description available.')}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all record set `@id`s present in the dataset
record_sets = list(dataset.record_sets)
if not record_sets:
    print("No record sets found in the dataset. Please check the Croissant schema or contact the dataset publisher.")
else:
    print("Record set @ids found:")
    for rs in record_sets:
        print(f"  - {rs['@id']}")

# Display fields (columns) for each record set
for rs in record_sets:
    print(f"\nRecord set: {rs['@id']}")
    if 'field' in rs:
        fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        print("  Fields and their @ids:")
        for field in fields:
            # field is either an @id reference (str or dict)
            if isinstance(field, dict) and '@id' in field:
                print(f"    - {field['@id']}")
            elif isinstance(field, str):
                print(f"    - {field}")
            else:
                print(f"    - {field}")
    else:
        print("  No fields found for this record set.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Collect record set IDs for extraction
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}
for record_set_id in record_set_ids:
    print(f"\nExtracting data from record set: {record_set_id}")
    try:
        df = pd.DataFrame(dataset.records(record_set=record_set_id))
        if not df.empty:
            dataframes[record_set_id] = df
            print(f"Fields: {df.columns.tolist()}")
            display(df.head())
        else:
            print(f"No records found in record set {record_set_id}.")
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

# Pick the first record set as an example for further EDA if available
main_record_set_id = record_set_ids[0] if record_set_ids else None
main_df = dataframes.get(main_record_set_id) if main_record_set_id else None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Example EDA: Filter and normalize a numeric field if present
import numpy as np
import warnings
warnings.filterwarnings('ignore')

if main_df is not None and not main_df.empty:
    # Attempt to pick a numeric field automatically (e.g., by checking dtype or known columns)
    numeric_field = None
    for col in main_df.columns:
        try:
            # Try to convert column to numeric to test
            _ = pd.to_numeric(main_df[col], errors='coerce')
            if main_df[col].dtype in [np.float64, np.int64] or pd.api.types.is_numeric_dtype(main_df[col]):
                numeric_field = col
                break
        except Exception:
            continue
    # If not, pick the first field
    if numeric_field is None:
        for col in main_df.columns:
            # try to coerce to float
            try:
                as_float = pd.to_numeric(main_df[col], errors='coerce')
                if as_float.notnull().any():
                    numeric_field = col
                    main_df[numeric_field] = as_float
                    break
            except Exception:
                pass

    if numeric_field:
        print(f"Using numeric field '{numeric_field}' for analysis.")
        # Set a threshold as an example
        threshold = main_df[numeric_field].mean() if main_df[numeric_field].dtype != object else 0
        filtered_df = main_df[pd.to_numeric(main_df[numeric_field], errors='coerce') > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by the first non-numeric column
        group_field = None
        for col in main_df.columns:
            if col != numeric_field and not pd.api.types.is_numeric_dtype(main_df[col]):
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True).reset_index()
            print(f"Grouped data by '{group_field}':")
            display(grouped_df.head())
        else:
            print("No suitable group field found.")
    else:
        print("No numeric fields found for EDA in the selected record set.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_df is not None and not main_df.empty and numeric_field:
    plt.figure(figsize=(8, 5))
    sns.histplot(main_df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # If a group field was found in EDA, plot the mean numeric field by group
    if 'group_field' in locals() and group_field and group_field in main_df.columns:
        plt.figure(figsize=(10, 5))
        plot_df = main_df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False)
        plot_df.plot(kind='bar')
        plt.title(f"Average {numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.tight_layout()
        plt.show()
else:
    print("No suitable dataframe or numeric field for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

*In this notebook, we've demonstrated end-to-end loading and exploration of the FAIR^2 dataset on human mast cell extracellular vesicle proteomics using the `mlcroissant` library. The dataset follows the Croissant format and allows programmatic access to its schema, metadata, and record content by referencing entities with their `@id`. With this framework, you can scale this workflow to new datasets, apply statistical and ML analysis, or extend visual reporting further as needed.*